# Channel aggregation

Binary FACED analysis using previously stored predictions; no retraining.

In [ ]:
import json,hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02')
out=root/'signal_analysis';out.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):h.update(part)
    return h.hexdigest()
def measure(y,p):
    y=np.asarray(y).reshape(-1);p=np.asarray(p).reshape(-1)
    assert y.shape==p.shape and np.isin(y,[0,1]).all() and np.isin(p,[0,1]).all()
    return {'accuracy':float(accuracy_score(y,p)),'balanced_accuracy':float(balanced_accuracy_score(y,p)),
            'macro_f1':float(f1_score(y,p,average='macro',zero_division=0))}


In [ ]:
# All four rows have identical split, seed, input features, graph and temporal model.
# The best epoch in each case was selected on this same validation split.
variants=[('Mean','A00001'),('Max','A00026'),('Attention','A00027'),('Hierarchical','A00028')]
records=[];reference_subjects=reference_videos=reference_labels=None;source_hashes={}
for name,run_id in variants:
    folder=root/'graph_experiments'/'runs'/run_id
    source=folder/'validation_predictions.npz';info=folder/'metrics.json';config=folder/'config.json'
    saved=np.load(source);metric=json.loads(info.read_text());cfg=json.loads(config.read_text())
    pred,y=saved['predictions'],saved['labels'];assert pred.shape==y.shape==(528,30)
    if reference_labels is None:
        reference_subjects=saved['subjects'].copy();reference_videos=saved['videos'].copy();reference_labels=y.copy()
    assert np.array_equal(saved['subjects'],reference_subjects) and np.array_equal(saved['videos'],reference_videos) and np.array_equal(y,reference_labels)
    assert cfg['fold']==0 and cfg['seed']==42 and cfg['epochs']==10 and cfg['train_subjects'] and len(cfg['validation_subjects'])==22
    score=measure(y,pred);assert np.isclose(score['balanced_accuracy'],metric['validation']['balanced_accuracy'])
    records.append({'method':name,'run_id':run_id,'subjects':len(np.unique(saved['subjects'])),'trials':len(pred),
                    'seconds':pred.size,'best_epoch':metric['best_epoch'],'parameters':metric['parameters'],**score})
    for p in [source,info,config]:source_hashes[str(p)]=sha256(p)
table=pd.DataFrame(records);table.to_csv(out/'channel_aggregation.csv',index=False)
summary={'comparison':'graph readout across 30 electrode nodes, same split and GAT-LSTM backbone',
         'evaluation':'one-second predictions on the same 22 validation subjects; best epoch selected on validation',
         'warning':'exploratory only; attention and hierarchical pooling use extra trainable parameters',
         'sources_sha256':source_hashes,'results':records}
(out/'channel_aggregation.json').write_text(json.dumps(summary,indent=2),encoding='utf8')
ax=table.plot.bar(x='method',y='balanced_accuracy',legend=False,ylim=(0.5,0.6),rot=0,title='Channel aggregation: validation balanced accuracy')
ax.set_ylabel('Balanced accuracy');ax.figure.tight_layout();ax.figure.savefig(out/'channel_aggregation.png',dpi=150);plt.show()
print(table.to_string(index=False));print('Saved to:',out)
